In [6]:
%%capture
!pip install facenet-pytorch

## Libraries

In [15]:
import numpy as np
from facenet_pytorch import MTCNN, InceptionResnetV1, fixed_image_standardization, training
import torch
import torch.nn as nn
from torchvision import models
from torchvision import transforms
from torchsummary import summary
from PIL import Image

## Functions

In [50]:
def model_size(model):
    size_model = 0
    for param in model.parameters():
        if param.data.is_floating_point():
            size_model += param.numel() * torch.finfo(param.data.dtype).bits
        else:
            size_model += param.numel() * torch.iinfo(param.data.dtype).bits
    print(f"model size: {size_model} / bit | {size_model / 8e6:.2f} / MB")

In [123]:
def detect_crop_image(path, model, transform, device):
    pil_image = Image.open(path)
    boxes, _ = mtcnn.detect(pil_image)
    boxes = boxes.astype(int)
    
    cropped_images = []
    for box in boxes:
        cropped_image = image.crop(box)
        cropped_images.append(transform(cropped_image))
    return torch.stack(cropped_images).to(device)

## Load models

In [124]:
# set device
device = torch.device('cuda:0' if torch.cuda.is_available() else 'cpu')
print('Current device: {}'.format(device))

Current device: cuda:0


In [125]:
# Load model
'''
This model is used to detect faces and it returns the face (cropped)

image_size: output image size
margin: margin of the bounding box added in the ouput image
min_face_size: minimum face size to search within the image
'''
mtcnn = MTCNN(
    image_size=160,
    margin=0,
    min_face_size=20,
    thresholds=[0.6, 0.7, 0.7],
    factor=0.709,
    post_process=False,
    device=device
)

In [126]:
model_size(mtcnn)

model size: 15867200 / bit | 1.98 / MB


In [127]:
# Load model
'''
The cropped faces are passed as input in the CNN and we get an embedding for each face.
Important to set the model at .eval()
'''
resnet = InceptionResnetV1(
    classify=True,
    pretrained='vggface2',
    num_classes=100
).to(device)


In [128]:
model_size(resnet)

model size: 753085568 / bit | 94.14 / MB


In [129]:
distill_model = models.mobilenet_v3_small(pretrained=True)

In [130]:
model_size(distill_model)

model size: 81371392 / bit | 10.17 / MB


In [131]:
summary(distill_model.to(device), (3, 160, 160)) # works with same input as resnet!!!

----------------------------------------------------------------
        Layer (type)               Output Shape         Param #
            Conv2d-1           [-1, 16, 80, 80]             432
       BatchNorm2d-2           [-1, 16, 80, 80]              32
         Hardswish-3           [-1, 16, 80, 80]               0
            Conv2d-4           [-1, 16, 40, 40]             144
       BatchNorm2d-5           [-1, 16, 40, 40]              32
              ReLU-6           [-1, 16, 40, 40]               0
 AdaptiveAvgPool2d-7             [-1, 16, 1, 1]               0
            Conv2d-8              [-1, 8, 1, 1]             136
              ReLU-9              [-1, 8, 1, 1]               0
           Conv2d-10             [-1, 16, 1, 1]             144
      Hardsigmoid-11             [-1, 16, 1, 1]               0
SqueezeExcitation-12           [-1, 16, 40, 40]               0
           Conv2d-13           [-1, 16, 40, 40]             256
      BatchNorm2d-14           [-1, 16,

In [132]:
path = '/home/pj00/projects/Github/small_face_recognition_trcking/Data/train_images/0/1.jpg'

In [133]:
transform = transforms.Compose([
            transforms.Resize((160, 160)),
            transforms.ToTensor(),
            transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
        ])

In [134]:
cropped_images = detect_crop_image(path, mtcnn, transform, device)